In [6]:
text = "Once upon a time, there was a little girl named Lily. She loved to play outside in the park."

chars = sorted(set(text))
vocab_size = len(chars)
print("vocab:", chars)
print("vocab_size:", vocab_size)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

ids = encode(text)
print("ids[:20]:", ids[:20])
print("문자 수:", len(text), "/ 토큰 수:", len(ids))
assert decode(ids) == text                     # 왕복(round-trip) 확인
print("round-trip OK")

vocab: [' ', ',', '.', 'L', 'O', 'S', 'a', 'c', 'd', 'e', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y']
vocab_size: 26
ids[:20]: [4, 16, 7, 9, 0, 22, 18, 17, 16, 0, 6, 0, 21, 12, 15, 9, 1, 0, 21, 11]
문자 수: 92 / 토큰 수: 92
round-trip OK


In [7]:
try:
    encode("Zebra")
except KeyError as e:
    print("KeyError:", e)

# ② 단어 단위로 쪼개면?
words = text.replace(".", " .").replace(",", " ,").split()
print("단어 토큰 수:", len(words), "/ 단어 vocab:", len(set(words)))
print(words)

KeyError: 'Z'
단어 토큰 수: 22 / 단어 vocab: 20
['Once', 'upon', 'a', 'time', ',', 'there', 'was', 'a', 'little', 'girl', 'named', 'Lily', '.', 'She', 'loved', 'to', 'play', 'outside', 'in', 'the', 'park', '.']


In [8]:
ids = list(text.encode("utf-8"))
print("초기 길이:", len(ids))

def get_stats(ids):
    """인접한 두 토큰 쌍의 등장 횟수 세기"""
    counts = {}
    for pair in zip(ids, ids[1:]):    # (ids[0],ids[1]), (ids[1],ids[2]), ...
        counts[pair] = counts.get(pair, 0) + 1
    return counts

stats = get_stats(ids)
top = sorted(stats.items(), key=lambda x: x[1], reverse=True)[:5]
for (a, b), cnt in top:
    print(f"({chr(a)!r}, {chr(b)!r}) -> {cnt}회")

초기 길이: 92
('e', ' ') -> 6회
(' ', 't') -> 4회
('h', 'e') -> 3회
('n', ' ') -> 2회
(' ', 'a') -> 2회


In [9]:
def merge(ids, pair, new_id):
    """ids에서 pair가 연속으로 나오는 곳을 new_id 하나로 교체"""
    out = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            out.append(new_id)        # 두 토큰 → 한 토큰
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out

# 작은 예제로 검증
print(merge([5, 6, 6, 7, 9, 1], (6, 7), 99))   # 기대: [5, 6, 99, 9, 1]

[5, 6, 99, 9, 1]


In [10]:
num_merges = 10
merges = {}                           # (a, b) -> new_id, 순서대로 저장됨
ids_work = list(ids)

for k in range(num_merges):
    stats = get_stats(ids_work)
    pair = max(stats, key=stats.get)  # 가장 많이 나온 쌍
    new_id = 256 + k
    ids_work = merge(ids_work, pair, new_id)
    merges[pair] = new_id
    print(f"merge {k+1}: {pair} -> {new_id}  (빈도 {stats[pair]}, 길이 {len(ids_work)})")

print(f"\n길이: {len(ids)} -> {len(ids_work)}, 압축률 {len(ids)/len(ids_work):.2f}x")

merge 1: (101, 32) -> 256  (빈도 6, 길이 86)
merge 2: (32, 116) -> 257  (빈도 4, 길이 82)
merge 3: (32, 97) -> 258  (빈도 2, 길이 80)
merge 4: (109, 101) -> 259  (빈도 2, 길이 78)
merge 5: (257, 104) -> 260  (빈도 2, 길이 76)
merge 6: (79, 110) -> 261  (빈도 1, 길이 75)
merge 7: (261, 99) -> 262  (빈도 1, 길이 74)
merge 8: (262, 256) -> 263  (빈도 1, 길이 73)
merge 9: (263, 117) -> 264  (빈도 1, 길이 72)
merge 10: (264, 112) -> 265  (빈도 1, 길이 71)

길이: 92 -> 71, 압축률 1.30x


In [12]:
# 토큰 ID → 바이트열 테이블
vocab = {i: bytes([i]) for i in range(256)}       # 기본 바이트 256개
for (a, b), idx in merges.items():                 # dict는 삽입 순서 유지 = 병합 순서
    vocab[idx] = vocab[a] + vocab[b]               # 두 조각의 바이트를 이어붙임

def decode(ids):
    raw = b"".join(vocab[i] for i in ids)          # 전체 바이트열
    return raw.decode("utf-8", errors="replace")   # 깨진 바이트는 � 로 대체

for idx in range(256, 256 + len(merges)):
    print(idx, vocab[idx])

256 b'e '
257 b' t'
258 b' a'
259 b'me'
260 b' th'
261 b'On'
262 b'Onc'
263 b'Once '
264 b'Once u'
265 b'Once up'


In [13]:
def encode(s):
    ids = list(s.encode("utf-8"))                  # 바이트 단위로 시작, 길이 = 바이트 수
    while len(ids) >= 2:
        stats = get_stats(ids)
        # 현재 등장한 쌍 중 "가장 먼저 학습된" 병합 고르기 (ID가 작을수록 먼저 학습됨)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:                     # 적용할 병합이 더 없음
            break
        ids = merge(ids, pair, merges[pair])
    return ids

def show(s):
    ids = encode(s)
    pieces = [vocab[i].decode("utf-8", errors="replace") for i in ids]
    print(f"{s!r}\n  -> {len(s.encode())}바이트 → {len(ids)}토큰: {pieces}")
    assert decode(ids) == s                        # 왕복 확인

show(text)
show("Once upon a time")
show("Once upon a mouse")
show("Zebra")

'Once upon a time, there was a little girl named Lily. She loved to play outside in the park.'
  -> 92바이트 → 71토큰: ['Once up', 'o', 'n', ' a', ' t', 'i', 'me', ',', ' th', 'e', 'r', 'e ', 'w', 'a', 's', ' a', ' ', 'l', 'i', 't', 't', 'l', 'e ', 'g', 'i', 'r', 'l', ' ', 'n', 'a', 'me', 'd', ' ', 'L', 'i', 'l', 'y', '.', ' ', 'S', 'h', 'e ', 'l', 'o', 'v', 'e', 'd', ' t', 'o', ' ', 'p', 'l', 'a', 'y', ' ', 'o', 'u', 't', 's', 'i', 'd', 'e ', 'i', 'n', ' th', 'e ', 'p', 'a', 'r', 'k', '.']
'Once upon a time'
  -> 16바이트 → 7토큰: ['Once up', 'o', 'n', ' a', ' t', 'i', 'me']
'Once upon a mouse'
  -> 17바이트 → 10토큰: ['Once up', 'o', 'n', ' a', ' ', 'm', 'o', 'u', 's', 'e']
'Zebra'
  -> 5바이트 → 5토큰: ['Z', 'e', 'b', 'r', 'a']


In [14]:
for s in ["a", "é", "안", "😀"]:
    b = s.encode("utf-8")
    print(f"{s!r}: {list(b)} → {len(b)}바이트")

s = "안녕하세요"
print(f"\n{s!r}: {len(s)}글자 → {len(s.encode('utf-8'))}바이트")

# 한 글자를 이루는 바이트 중 하나만 디코딩하면?
print("'안'의 첫 바이트만:", decode([list('안'.encode('utf-8'))[0]]))

'a': [97] → 1바이트
'é': [195, 169] → 2바이트
'안': [236, 149, 136] → 3바이트
'😀': [240, 159, 152, 128] → 4바이트

'안녕하세요': 5글자 → 15바이트
'안'의 첫 바이트만: �


In [17]:
import regex as re

# GPT-2 사전 분할 패턴
gpt2_pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

sample = "Once upon a time, there was a little girl. She's 3 years old!!"
print(gpt2_pat.findall(sample))

['Once', ' upon', ' a', ' time', ',', ' there', ' was', ' a', ' little', ' girl', '.', ' She', "'s", ' 3', ' years', ' old', '!!']


In [18]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")
print("vocab size:", enc.n_vocab)

def show_gpt2(s):
    ids = enc.encode(s)
    pieces = [enc.decode([i]) for i in ids]
    print(f"{s!r}: {len(ids)}토큰 {ids} {pieces}")

for s in ["Once upon a time", "once upon a time", " Once", "Once upon a mouse",
          "Zebra", "replaying", "안녕하세요"]:
    show_gpt2(s)

# 특수 토큰: 문서(이야기)의 끝을 표시
print("EOT id:", enc.eot_token)
print(enc.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

vocab size: 50257
'Once upon a time': 4토큰 [7454, 2402, 257, 640] ['Once', ' upon', ' a', ' time']
'once upon a time': 4토큰 [27078, 2402, 257, 640] ['once', ' upon', ' a', ' time']
' Once': 1토큰 [4874] [' Once']
'Once upon a mouse': 4토큰 [7454, 2402, 257, 10211] ['Once', ' upon', ' a', ' mouse']
'Zebra': 2토큰 [57, 37052] ['Z', 'ebra']
'replaying': 2토큰 [260, 17916] ['re', 'playing']
'안녕하세요': 14토큰 [168, 243, 230, 167, 227, 243, 47991, 246, 168, 226, 116, 168, 248, 242] ['�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�']
EOT id: 50256
[50256]


In [5]:
from transformers import AutoTokenizer

hf_tok = AutoTokenizer.from_pretrained("gpt2")
s = "Once upon a time, there was a little girl named Lily. She's 3 years old!!"
a = enc.encode(s)
b = hf_tok.encode(s)
print("tiktoken:", a)
print("HF      :", b)
print("same:", a == b)

tiktoken: [7454, 2402, 257, 640, 11, 612, 373, 257, 1310, 2576, 3706, 20037, 13, 1375, 338, 513, 812, 1468, 3228]
HF      : [7454, 2402, 257, 640, 11, 612, 373, 257, 1310, 2576, 3706, 20037, 13, 1375, 338, 513, 812, 1468, 3228]
same: True


In [19]:
vocab_size, n_embd, n_layer, block_size = 50257, 384, 6, 256

wte = vocab_size * n_embd            # 토큰 임베딩 (50257, 384)
wpe = block_size * n_embd            # 위치 임베딩 (256, 384)
per_layer = 12 * n_embd**2           # 블록 1개의 대략적인 가중치 수 (bias, LayerNorm 제외)
blocks = n_layer * per_layer

total = wte + wpe + blocks
print(f"토큰 임베딩: {wte:>12,} ({wte/total:.0%})")
print(f"위치 임베딩: {wpe:>12,}")
print(f"블록 {n_layer}개:   {blocks:>12,} ({blocks/total:.0%})")
print(f"합계(대략):  {total:>12,}")

토큰 임베딩:   19,298,688 (64%)
위치 임베딩:       98,304
블록 6개:     10,616,832 (35%)
합계(대략):    30,013,824


In [16]:
import torch
import torch.nn as nn

emb = nn.Embedding(50257, 384)                    # 가중치 행렬 (50257, 384)
print("weight:", emb.weight.shape)

ids = torch.tensor([enc.encode("Once upon a time")])   # (B=1, T=토큰 수)
x = emb(ids)                                      # (B, T, C=384)
print("ids:", ids.shape, "→ x:", x.shape)

# 임베딩 = 그 ID의 행을 꺼내오는 것
print(torch.equal(x[0, 0], emb.weight[ids[0, 0]]))     # True
print(ids[0,0])

weight: torch.Size([50257, 384])
ids: torch.Size([1, 4]) → x: torch.Size([1, 4, 384])
True
tensor(7454)
